# GraphTrust reproducible heavy-compute workflow
This notebook runs CPU/RAM graph analytics. It does not claim GPU acceleration and it never changes live permissions. Reruns use checksum-verified checkpoints.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("GRAPHTRUST_REPO_URL", "https://github.com/sronters/graph.git")
MOUNT_DRIVE = os.getenv("GRAPHTRUST_MOUNT_DRIVE", "0") == "1"
FULL_MATRIX = os.getenv("GRAPHTRUST_FULL_MATRIX", "1") == "1"
print(
    {
        "python": sys.version,
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "repo": REPO_URL,
    }
)
if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

## Acquire the repository
A public clone is preferred. If `REPO_URL` is empty, upload a repository ZIP; no credential is embedded.

In [ ]:
root = Path("/content/graphtrust")
if not (root / "pyproject.toml").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    else:
        from google.colab import files

        uploaded = files.upload()
        archive = next(Path("/content") / name for name in uploaded if name.endswith(".zip"))
        shutil.unpack_archive(archive, root)
os.chdir(root)
print(root)

## Install locked dependencies and record runtime capacity

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--all-extras", "--frozen"], check=True)
import psutil  # noqa: E402

memory_gib = psutil.virtual_memory().available / 2**30
workers = max(1, min(2, (os.cpu_count() or 1) // 2))
print({"available_ram_gib": round(memory_gib, 2), "deterministic_workers": workers})
if memory_gib < 12:
    print(
        "Large generation is disabled below 12 GiB available RAM; "
        "medium checkpoints remain runnable."
    )

## Validate code provenance and configuration hashes

In [ ]:
import hashlib

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
hashes = {
    name: hashlib.sha256(Path(name).read_bytes()).hexdigest()
    for name in ["uv.lock", "configs/default.yaml", "configs/experiments.yaml"]
}
subprocess.run(["uv", "run", "graphtrust", "--version"], check=True)
print({"git_commit": commit, "hashes": hashes})

## Generate or load the complete medium matrix
Generation is deterministic and skips directories that already contain a checksum inventory.

In [ ]:
profiles = ["saas_scaleup", "regulated_finance", "global_hybrid"]
final_seeds = [2750159, 3681131, 4264387, 5000011, 6500027]
variants = "clean,injected_low,injected_mixed,remediated_truth"
if FULL_MATRIX:
    for profile in profiles:
        for seed in final_seeds:
            target = Path(f"data/generated/{profile}/medium/{seed}/injected_mixed/checksums.sha256")
            if not target.exists():
                subprocess.run(
                    [
                        "uv",
                        "run",
                        "graphtrust",
                        "generate",
                        "--profile",
                        profile,
                        "--scale",
                        "medium",
                        "--seed",
                        str(seed),
                        "--variants",
                        variants,
                    ],
                    check=True,
                )

## Correctness parity before heavy inference

In [ ]:
subprocess.run(
    [
        "uv",
        "run",
        "pytest",
        "tests/unit/test_networkx_backend.py",
        "tests/integration/test_analysis_pipeline.py",
        "-q",
    ],
    check=True,
)

## Run the full medium experiment matrix with verified resume

In [ ]:
if FULL_MATRIX:
    subprocess.run(
        [
            "uv",
            "run",
            "graphtrust",
            "experiment",
            "--registry",
            "configs/experiments.yaml",
            "--workers",
            str(workers),
            "--resume",
        ],
        check=True,
    )

## Large datasets, one profile at a time
Large graphs are attempted only when memory is sufficient. Each profile is released before the next.

In [ ]:
import gc

large_seed = final_seeds[0]
if FULL_MATRIX and memory_gib >= 12:
    for profile in profiles:
        target = Path(
            f"data/generated/{profile}/large/{large_seed}/injected_mixed/checksums.sha256"
        )
        if not target.exists():
            subprocess.run(
                [
                    "uv",
                    "run",
                    "graphtrust",
                    "generate",
                    "--profile",
                    profile,
                    "--scale",
                    "large",
                    "--seed",
                    str(large_seed),
                    "--variants",
                    variants,
                ],
                check=True,
            )
        gc.collect()

## Large GraphTrust and selected baselines
This uses the same immutable run writer as the registered experiment matrix.

In [ ]:
if FULL_MATRIX and memory_gib >= 12:
    from graphtrust.data import read_dataset
    from graphtrust.experiments.registry import ExperimentUnit
    from graphtrust.experiments.runner import run_experiment_unit
    from graphtrust.schemas.findings import AnalysisMethod
    from graphtrust.schemas.manifests import DatasetVariant
    from graphtrust.settings import load_project_config

    config = load_project_config()
    for profile in profiles:
        for variant in (
            DatasetVariant.CLEAN,
            DatasetVariant.INJECTED_LOW,
            DatasetVariant.INJECTED_MIXED,
        ):
            path = Path(f"data/generated/{profile}/large/{large_seed}/{variant.value}")
            bundle = read_dataset(path)
            for method in AnalysisMethod:
                unit = ExperimentUnit(
                    path, bundle.manifest.dataset_id, profile, "large", large_seed, variant, method
                )
                run_experiment_unit(unit, config, resume=True)
            gc.collect()

## Remediation, ablation, and sensitivity evaluation

In [ ]:
if FULL_MATRIX:
    sample = Path(f"data/generated/saas_scaleup/medium/{final_seeds[0]}/injected_mixed")
    subprocess.run(
        [
            "uv",
            "run",
            "graphtrust",
            "analyze",
            "--dataset",
            str(sample),
            "--methods",
            "graphtrust",
            "--config",
            "configs/default.yaml",
        ],
        check=True,
    )
    subprocess.run(
        [
            "uv",
            "run",
            "graphtrust",
            "remediate",
            "--analysis",
            "artifacts/latest",
            "--solvers",
            "degree_greedy,risk_greedy,min_cut,constraint_generation",
            "--targets",
            "0.80,0.90,1.00",
        ],
        check=True,
    )
    subprocess.run(
        [
            "uv",
            "run",
            "python",
            "scripts/run_extended_evaluation.py",
            "--dataset",
            str(sample),
            "--output",
            "artifacts/extended",
        ],
        check=True,
    )

## Produce figures, tables, and verify traceability

In [ ]:
subprocess.run(
    [
        "uv",
        "run",
        "graphtrust",
        "report",
        "--runs",
        "artifacts/manifests",
        "--output",
        "artifacts/paper",
    ],
    check=True,
)
subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/generate_figures.py",
        "--runs",
        "artifacts/manifests",
        "--output",
        "artifacts/paper/figures",
    ],
    check=True,
)
subprocess.run(
    ["uv", "run", "python", "scripts/validate_traceability.py", "--artifacts", "artifacts"],
    check=True,
)

## Package artifacts and optionally copy to Drive

In [ ]:
from datetime import UTC, datetime

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
archive = shutil.make_archive(f"/content/graphtrust-artifacts-{stamp}", "zip", "artifacts")
if MOUNT_DRIVE:
    shutil.copy2(archive, "/content/drive/MyDrive/")
print(archive)